<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/06-numpy/01_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6: NumPy

For five modules we've done everything with plain Python lists and loops. It works, and you now know exactly what's happening inside a k-NN classifier, a scaler or a training loop. But it's slow. In Module 3, training a tiny neural network on 60 points took several seconds.

NumPy fixes that. It gives you **arrays**: blocks of numbers that all have the same type, stored side by side in memory, with operations written in C that work on the whole array at once. Almost every numerical library in Python is built on it, from pandas and scikit-learn to the ideas behind PyTorch tensors, which deliberately work the same way.

The big shift in this module is a way of thinking: instead of writing a loop that handles one number at a time, you write one expression that handles all of them. That's called **vectorization**, and once it clicks, ML code becomes both shorter and dramatically faster.

**Contents**

1. Why NumPy is fast
2. Creating arrays
3. Shape: the most important property of an array
4. Indexing, slicing, and the view trap
5. Vectorized operations and axes
6. Broadcasting
7. dtypes and numerical pitfalls
8. Linear algebra
9. Random numbers, the modern way
10. Putting it together: k-NN without loops

## 1. Why NumPy is fast

Let's measure it. The sum of squares of a million numbers, in plain Python and in NumPy:

In [1]:
import time
import numpy as np

n = 1_000_000
values_list = list(range(n))
values_array = np.arange(n, dtype=np.float64)

def best_time(func, repeats=5):
    """Run func several times and return the fastest time, in seconds."""
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        result = func()
        times.append(time.perf_counter() - start)
    return min(times), result


t_py, total_py = best_time(lambda: sum(v * v for v in values_list))
t_np, total_np = best_time(lambda: np.sum(values_array * values_array))

print(f"python: {t_py * 1000:7.1f} ms   result {total_py:.4e}")
print(f"numpy:  {t_np * 1000:7.1f} ms   result {total_np:.4e}")
print(f"numpy is about {t_py / t_np:.0f}x faster")

python:    39.9 ms   result 3.3333e+17
numpy:      1.1 ms   result 3.3333e+17
numpy is about 37x faster


Same answer, and NumPy is many times faster. The exact ratio depends on your machine and on the operation: for a simple sum like this it's often 10 to 40 times, and for heavier work, like the k-NN at the end of this lesson, the gap gets bigger.

Notice that each version runs five times and we keep the **fastest** time. The very first NumPy call in a session can be several times slower than later ones, while memory is set up and libraries load, so a single measurement can make NumPy look no faster than plain Python. Taking the best of several runs is the standard way to time code; Module 10 comes back to this.

Why is it faster at all?

- **A Python list holds pointers to separate objects.** Every number is a full Python object with a type, a reference count and the value itself, scattered around memory. Every `v * v` has to look up the type, find the right multiplication code, and create a new object for the result.
- **A NumPy array holds raw numbers in one contiguous block**, all with the same type. Multiplying two arrays runs one tight loop in compiled C over that block, with no type checks and no new objects per element. Modern CPUs can even process several numbers in a single instruction.

The catch is that this only helps if you let NumPy do the looping. If you write a Python `for` loop over a NumPy array, element by element, you get Python speed again, or worse. The rest of this module is about how to express things without those loops.

## 2. Creating arrays

In [2]:
a = np.array([1.5, 2.0, 3.5])
print(a, a.dtype)

b = np.array([[1, 2, 3],
              [4, 5, 6]])
print(b)
print(b.dtype)

[1.5 2.  3.5] float64
[[1 2 3]
 [4 5 6]]
int64


Every array has a single **dtype** (data type). `a` holds 64-bit floats, `b` holds 64-bit integers, because every value we gave it was an integer. Mixing types makes NumPy pick one that fits everything:

In [3]:
print(np.array([1, 2.5]).dtype)        # ints and floats -> float
print(np.array([1, "2"]).dtype)        # numbers and text -> text (careful!)

float64
<U21


That second one is a classic surprise: one stray string turns the whole array into strings, and then arithmetic fails. It happens when you build an array from a CSV column you forgot to convert, as in Module 5.

Arrays of a given size, filled in standard ways:

In [4]:
print(np.zeros(3))
print(np.ones((2, 3)))
print(np.full((2, 2), 7.0))
print(np.eye(3))                      # identity matrix
print(np.arange(0, 10, 2))            # like range(), but an array
print(np.linspace(0, 1, 5))           # 5 evenly spaced points from 0 to 1, end included

[0. 0. 0.]
[[1. 1. 1.]
 [1. 1. 1.]]
[[7. 7.]
 [7. 7.]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


Use `arange` for integer steps and `linspace` when you want an exact number of points between two values. `np.arange(0, 1, 0.1)` with a float step can give one more or one fewer element than you expect, because of the float rounding from Module 1.

Converting back to Python types is sometimes needed, for example before saving to JSON (which doesn't know about NumPy types):

In [5]:
print(a.tolist())
print(float(a.sum()), type(float(a.sum())))

[1.5, 2.0, 3.5]
7.0 <class 'float'>


## 3. Shape

Every array has a **shape**: a tuple giving its size along each dimension (also called an *axis*). Shape is the single most important property of an array. Most NumPy bugs are shape bugs, and most of the time, reading the shapes tells you exactly what a piece of ML code does.

In [6]:
X = np.arange(12).reshape(3, 4)
print(X)
print("shape:", X.shape)    # 3 rows, 4 columns
print("ndim: ", X.ndim)     # number of dimensions
print("size: ", X.size)     # total number of elements

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
shape: (3, 4)
ndim:  2
size:  12


In ML, the standard convention is that a dataset is a 2D array of shape `(n_samples, n_features)`: one row per example, one column per feature. Images add more axes: a batch of colour images in PyTorch is `(batch, channels, height, width)`.

`reshape` changes the shape without changing the data, as long as the total size stays the same. Passing `-1` for one dimension means "work this one out for me":

In [7]:
v = np.arange(6)
print(v.reshape(2, 3))
print(v.reshape(3, -1))      # 3 rows, columns = 6 / 3
print(v.reshape(-1, 1))      # a column: shape (6, 1)

try:
    v.reshape(4, 2)
except ValueError as e:
    print("ValueError:", e)

[[0 1 2]
 [3 4 5]]
[[0 1]
 [2 3]
 [4 5]]
[[0]
 [1]
 [2]
 [3]
 [4]
 [5]]
ValueError: cannot reshape array of size 6 into shape (4,2)


### 1D vs 2D: a common confusion

An array of shape `(3,)` and one of shape `(3, 1)` or `(1, 3)` hold the same three numbers, but they behave differently:

In [8]:
v = np.array([1, 2, 3])
col = v.reshape(-1, 1)
row = v.reshape(1, -1)

print(v.shape, col.shape, row.shape)
print((col + row).shape)       # broadcasting (section 6) turns these into a 3x3 table
print(col + row)

(3,) (3, 1) (1, 3)
(3, 3)
[[2 3 4]
 [3 4 5]
 [4 5 6]]


`v[:, None]` (or `v[:, np.newaxis]`) is a common shorthand for adding an axis, turning shape `(3,)` into `(3, 1)`:

In [9]:
print(v[:, None].shape, v[None, :].shape)

(3, 1) (1, 3)


Transposing swaps the axes, so rows become columns. `.T` is the shortcut:

In [10]:
print(X.T.shape)
print(X.T)

(4, 3)
[[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]


## 4. Indexing, slicing, and the view trap

Indexing works like nested lists, but with all the indexes in one pair of brackets, separated by commas:

In [11]:
X = np.arange(12).reshape(3, 4)
print(X)
print(X[1, 2])         # row 1, column 2
print(X[0])            # the first row
print(X[:, 1])         # the second column (all rows, column 1)
print(X[1:, ::2])      # rows from 1 on, every second column
print(X[-1, -1])       # last row, last column

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
6
[0 1 2 3]
[1 5 9]
[[ 4  6]
 [ 8 10]]
11


`X[:, 1]`, meaning "every row, column 1", is something nested lists can't do in one step. You'll use it constantly to pull out a feature column.

### Slices are views, not copies

This is the most important difference from Python lists, and the source of some very confusing bugs. Slicing a list makes a copy. **Slicing an array makes a view**: a new array object that looks at the *same memory*. Changing the view changes the original:

In [12]:
data = np.arange(6, dtype=float)
first_three = data[:3]
first_three[0] = 999

print(data)        # the original changed!

[999.   1.   2.   3.   4.   5.]


NumPy does this on purpose: copying a large array is expensive, and views make slicing almost free. But it's the aliasing trap from Module 1 again, now hiding inside an operation that copies for lists. A real example: you take `X_train = X[:800]`, then scale `X_train` in place, and you've quietly scaled the first 800 rows of `X` too.

When you need an independent array, ask for one with `.copy()`:

In [13]:
data = np.arange(6, dtype=float)
first_three = data[:3].copy()
first_three[0] = 999
print(data)

print(np.shares_memory(data, data[:3]), np.shares_memory(data, data[:3].copy()))

[0. 1. 2. 3. 4. 5.]
True False


### Boolean masks

You can index with an array of `True`/`False` values of the same shape, which picks out the elements where the mask is `True`. Combined with comparisons, which produce boolean arrays, this replaces most `if` statements inside loops:

In [14]:
scores = np.array([0.2, 0.9, 0.55, 0.1, 0.75])

mask = scores > 0.5
print(mask)
print(scores[mask])
print(scores[(scores > 0.15) & (scores < 0.8)])   # & for "and", | for "or", ~ for "not"
print(mask.sum(), mask.mean())                    # True counts as 1

[False  True  True False  True]
[0.9  0.55 0.75]
[0.2  0.55 0.75]
3 0.6


Note the `&` instead of `and`, and the parentheses around each comparison. `and` doesn't work on arrays (it raises an error about "the truth value of an array"), and `&` binds more tightly than `>`, so without the parentheses Python would read it wrong.

`mask.mean()` is the fraction of `True` values. Accuracy, in NumPy, is just `(y_pred == y_true).mean()`.

Masks also work for assignment, which is a neat way to clip or fix values:

In [15]:
x = np.array([3.0, -1.0, 5.0, -2.0])
x[x < 0] = 0.0          # this is ReLU
print(x)

[3. 0. 5. 0.]


`np.where(condition, a, b)` is the vectorized `a if condition else b`:

In [16]:
probs = np.array([0.3, 0.8, 0.5, 0.1])
print(np.where(probs >= 0.5, 1, 0))

[0 1 1 0]


### Fancy indexing

Indexing with an array of integers picks those positions, in that order. It's how you shuffle a dataset, select a batch, or look up rows:

In [17]:
X = np.arange(10, 60, 10)
idx = np.array([4, 0, 2])
print(X[idx])

labels = np.array([0, 2, 1, 0])
class_names = np.array(["Adelie", "Chinstrap", "Gentoo"])
print(class_names[labels])          # map ids to names in one step

[50 10 30]
['Adelie' 'Gentoo' 'Chinstrap' 'Adelie']


Unlike slices, boolean and fancy indexing always return **copies**. That's the full rule: basic slices (`a:b:c`) give views, and everything else gives copies. If you're not sure, `np.shares_memory` will tell you.

## 5. Vectorized operations and axes

Arithmetic operators and math functions work element by element across whole arrays:

In [18]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([10.0, 20.0, 30.0])

print(a + b, a * b, b / a, a ** 2)
print(np.sqrt(a), np.exp(a), np.log(b))
print(np.maximum(a, 2.5))      # element-wise max: another way to write clipping

[11. 22. 33.] [10. 40. 90.] [10. 10. 10.] [1. 4. 9.]
[1.         1.41421356 1.73205081] [ 2.71828183  7.3890561  20.08553692] [2.30258509 2.99573227 3.40119738]
[2.5 2.5 3. ]


These element-wise functions are called **ufuncs** (universal functions), and they're the building blocks of vectorized code. A full sigmoid, for every element of an array at once, is one line:

In [19]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print(sigmoid(np.array([-5.0, 0.0, 5.0])))

[0.00669285 0.5        0.99330715]


### Reductions and the axis argument

Functions like `sum`, `mean`, `max` and `argmax` *reduce* an array to fewer numbers. By default they reduce everything to one number. The `axis` argument reduces along one dimension only:

In [20]:
X = np.array([[1, 2, 3],
              [4, 5, 6]])

print(X.sum())          # everything: 21
print(X.sum(axis=0))    # down the rows: one total per column
print(X.sum(axis=1))    # across the columns: one total per row

21
[5 7 9]
[ 6 15]


The way to remember it: **`axis` is the dimension that disappears**. `X` has shape `(2, 3)`. Summing over `axis=0` removes the first dimension, leaving shape `(3,)`: one number per column. Summing over `axis=1` leaves shape `(2,)`: one number per row.

For a dataset of shape `(n_samples, n_features)`, `axis=0` gives you per-feature statistics, which is what you want for scaling:

In [21]:
data = np.array([[150.0, 50.0],
                 [160.0, 65.0],
                 [170.0, 80.0]])

print("mean per feature:", data.mean(axis=0))
print("std per feature: ", data.std(axis=0))
print("max per sample:  ", data.max(axis=1))

mean per feature: [160.  65.]
std per feature:  [ 8.16496581 12.24744871]
max per sample:   [150. 160. 170.]


`argmax` returns the *position* of the largest value instead of the value itself. Along `axis=1` of a score matrix, that's a model's prediction for every sample at once:

In [22]:
class_scores = np.array([[0.1, 0.7, 0.2],
                         [0.8, 0.1, 0.1],
                         [0.3, 0.3, 0.4]])
print(class_scores.argmax(axis=1))

[1 0 2]


`keepdims=True` keeps the reduced axis with size 1, which is handy when the result has to be combined with the original array again, as the next section shows:

In [23]:
print(class_scores.sum(axis=1).shape, class_scores.sum(axis=1, keepdims=True).shape)

(3,) (3, 1)


## 6. Broadcasting

What happens when you combine arrays of different shapes? You've already seen one case: `array * 2` multiplies every element by 2. NumPy "stretches" the single number to match the array, without actually copying it. **Broadcasting** generalises this to arrays.

The rule: compare the shapes **from the right**. Two dimensions are compatible if they're equal, or if one of them is 1. A dimension of size 1 is stretched to match the other. If one array has fewer dimensions, it's treated as having extra size-1 dimensions on the left.

| shape A | shape B | result |
|---|---|---|
| `(3, 4)` | `(4,)` | `(3, 4)`: B is reused for every row |
| `(3, 4)` | `(3, 1)` | `(3, 4)`: B is reused for every column |
| `(3, 1)` | `(1, 4)` | `(3, 4)`: both are stretched |
| `(3, 4)` | `(3,)` | error: compare 4 with 3 |

The most common use in ML is standardizing features. `X` has shape `(n, d)`, and its mean and standard deviation along `axis=0` have shape `(d,)`, so they broadcast across every row:

In [24]:
rng = np.random.default_rng(0)
X = rng.normal(loc=[170.0, 70.0], scale=[10.0, 15.0], size=(1000, 2))   # height, weight

mean = X.mean(axis=0)
std = X.std(axis=0)
X_scaled = (X - mean) / std

print("shapes:", X.shape, mean.shape, X_scaled.shape)
print("new means:", X_scaled.mean(axis=0).round(6))
print("new stds: ", X_scaled.std(axis=0).round(6))

shapes: (1000, 2) (2,) (1000, 2)
new means: [-0. -0.]
new stds:  [1. 1.]


That's the entire `StandardScaler` from Module 3, applied to every feature of every sample, in one line, with no loops.

Now the last row of that table: the error case. Suppose you want to subtract each *row's* mean instead (centering each sample). `X.mean(axis=1)` has shape `(1000,)`, and broadcasting compares it with the *last* dimension of `X`, which is 2:

In [25]:
try:
    X - X.mean(axis=1)
except ValueError as e:
    print("ValueError:", e)

row_centered = X - X.mean(axis=1, keepdims=True)     # shape (1000, 1) broadcasts across columns
print(row_centered.shape)

ValueError: operands could not be broadcast together with shapes (1000,2) (1000,) 
(1000, 2)


`keepdims=True` gives shape `(1000, 1)`, which lines up with the rows. When you get a broadcasting error, print the shapes of both arrays and line them up from the right. That almost always shows the problem.

### Broadcasting to build a table

When one array has shape `(n, 1)` and the other `(1, m)`, broadcasting produces every combination, an `(n, m)` table. For example, the distance from every test point to every training point, which is exactly what k-NN needs. With points in `d` dimensions, we add axes so the shapes line up:

In [26]:
train_pts = rng.normal(size=(5, 2))    # 5 points in 2D
test_pts = rng.normal(size=(3, 2))     # 3 points in 2D

diff = test_pts[:, None, :] - train_pts[None, :, :]   # (3, 1, 2) - (1, 5, 2) -> (3, 5, 2)
dists = np.sqrt((diff ** 2).sum(axis=2))              # sum over the coordinate axis -> (3, 5)

print(diff.shape, dists.shape)
print(dists.round(2))

(3, 5, 2) (3, 5)
[[1.79 3.41 3.81 3.7  2.96]
 [0.83 1.52 1.34 1.24 0.52]
 [2.64 3.   1.48 1.36 1.89]]


Read the shapes and you can follow exactly what happens: one difference vector for each (test, train) pair, then square, sum over coordinates, square-root. Row `i` of `dists` holds the distances from test point `i` to all five training points. Section 10 uses this to build k-NN with no loops at all.

One warning: the intermediate `diff` array has `n_test × n_train × d` elements. For 10,000 test points, 100,000 training points and 100 features, that's 100 billion numbers, far too many for memory. Section 10 shows a more memory-friendly way.

## 7. dtypes and numerical pitfalls

NumPy's fixed-size types are what make it fast, but they also bring problems that Python's own numbers don't have.

### Integer overflow

Python integers can grow forever (Module 1). NumPy integers have a fixed number of bits, and when a result doesn't fit, it **wraps around**, silently:

In [27]:
small = np.array([100, 120], dtype=np.int8)     # int8 holds -128 to 127
print(small + small)

pixels = np.array([200, 250], dtype=np.uint8)   # images are often stored as uint8, 0 to 255
print(pixels + 100)                             # wraps past 255
print(pixels.astype(np.float32) + 100)          # convert first

[-56 -16]
[44 94]
[300. 350.]


No error, no warning, just wrong numbers. This bites people working with images, which are usually loaded as `uint8`. Adding brightness or averaging two images without converting to float first gives garbage. The standard first step with image data is `image.astype(np.float32) / 255.0`.

### float32 vs float64

NumPy defaults to 64-bit floats. Deep learning mostly uses 32-bit (`float32`), and increasingly 16-bit, because they use less memory and GPUs are much faster with them. The price is precision:

In [28]:
print(np.float64(1.0) + np.float64(1e-10))
print(np.float32(1.0) + np.float32(1e-10))      # the tiny value is lost entirely
print(np.zeros(1_000_000, dtype=np.float64).nbytes, "bytes vs", np.zeros(1_000_000, dtype=np.float32).nbytes)

1.0000000001
1.0
8000000 bytes vs 4000000


A common source of errors when moving data into PyTorch is dtype mismatch: NumPy gives you `float64`, the model's weights are `float32`, and the operation fails. Converting explicitly with `.astype(np.float32)` avoids it.

### NaN spreads

Any arithmetic with `NaN` gives `NaN`, so a single missing value can wipe out a whole statistic. NumPy has `nan`-aware versions of the common reductions:

In [29]:
with_gap = np.array([1.0, 2.0, np.nan, 4.0])
print(with_gap.mean())
print(np.nanmean(with_gap))
print(np.isnan(with_gap))

nan
2.3333333333333335
[False False  True False]


### Comparing floats

The Module 1 rule still applies: don't compare floats with `==`. NumPy's versions of `math.isclose` work on whole arrays:

In [30]:
a = np.array([0.1 + 0.2, 1.0])
b = np.array([0.3, 1.0])
print(a == b)
print(np.isclose(a, b))
print(np.allclose(a, b))       # True if every element is close

[False  True]
[ True  True]
True


`np.allclose` is the right way to check that two implementations agree, for example a loop version and a vectorized version. The exercises use it in every check.

## 8. Linear algebra

Most of machine learning is linear algebra underneath. The key operation is matrix multiplication, written with `@`:

In [31]:
W = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])      # shape (3, 2)
x = np.array([1.0, -1.0])      # shape (2,)

print(W @ x)                   # shape (3,)

[-1. -1. -1.]


**The shape rule:** `(n, k) @ (k, m)` gives `(n, m)`. The inner dimensions must match, and they disappear. Shapes tell you everything. Here's a linear layer applied to a whole batch at once:

In [32]:
batch = rng.normal(size=(32, 4))        # 32 samples, 4 features each
weights = rng.normal(size=(4, 3))       # 4 inputs -> 3 outputs
bias = np.zeros(3)

outputs = batch @ weights + bias        # (32, 4) @ (4, 3) -> (32, 3), then bias broadcasts
print(outputs.shape)

(32, 3)


That one line is what `nn.Linear` does in PyTorch. Compare it to the `Neuron` and `Layer` classes from the Module 3 mini project: a loop over neurons, each with a loop over inputs, run once per sample. Here it's a single matrix multiplication for the whole batch.

`*` is element-wise multiplication, **not** matrix multiplication. Mixing them up is a classic bug, and broadcasting sometimes makes the wrong one run without an error.

### Solving linear regression directly

In Module 1 we fitted a line with gradient descent. For linear regression, there's also an exact solution. Stack the features into a matrix `X` with a column of ones for the bias, and the best weights `w` satisfy:

$$X^\top X \, w = X^\top y$$

`np.linalg.solve` solves this kind of equation directly:

In [33]:
xs = np.linspace(0, 5, 50)
ys = 2 * xs + 1 + rng.normal(0, 0.3, size=50)        # the true line is y = 2x + 1

X_design = np.column_stack([xs, np.ones_like(xs)])  # columns: x, and 1 for the bias
w = np.linalg.solve(X_design.T @ X_design, X_design.T @ ys)
print("slope, intercept:", w.round(3))

w_lstsq, *_ = np.linalg.lstsq(X_design, ys, rcond=None)
print("lstsq:           ", w_lstsq.round(3))

slope, intercept: [1.958 1.104]
lstsq:            [1.958 1.104]


One line, no learning rate, no epochs, and it lands on the same kind of answer as Module 1's 500 steps of gradient descent. `np.linalg.lstsq` solves the same problem in a more numerically stable way and is what you'd use in practice.

You might see code that computes `np.linalg.inv(X.T @ X) @ X.T @ y`. Avoid it: computing a matrix inverse is slower and less accurate than solving the equation directly.

Why do we bother with gradient descent at all, then? Because exact solutions like this only exist for a few simple models. Neural networks have no closed-form answer, so they're trained iteratively, as the mini project does.

### Norms

In [34]:
v = np.array([3.0, 4.0])
print(np.linalg.norm(v))                       # length of the vector
print(np.linalg.norm(rng.normal(size=(4, 3)), axis=1).round(3))  # length of each row

5.0
[2.933 1.432 2.144 0.815]


## 9. Random numbers, the modern way

NumPy's recommended way to make random numbers is to create a **generator** with a seed, and call methods on it:

In [35]:
rng = np.random.default_rng(seed=42)

print(rng.random(3))                       # uniform in [0, 1)
print(rng.normal(loc=0, scale=1, size=3))  # Gaussian
print(rng.integers(0, 10, size=5))         # integers from 0 to 9
print(rng.choice(["a", "b", "c"], size=4)) # random picks
print(rng.permutation(6))                  # a shuffled 0..5

[0.77395605 0.43887844 0.85859792]
[ 0.94056472 -1.95103519 -1.30217951]
[7 7 7 7 5]
['a' 'c' 'b' 'b']
[0 5 4 2 1 3]


You'll also see older code using `np.random.seed(42)` and `np.random.rand(...)`. That style uses one hidden global generator, so any function anywhere that draws a random number changes what every other function gets. A generator object passed around explicitly is the same idea as `random.Random(seed)` from earlier modules, and it keeps experiments reproducible.

`permutation` is how you shuffle a dataset: shuffle the **indexes**, then use fancy indexing so features and labels stay matched up:

In [36]:
X = np.arange(20).reshape(10, 2)
y = np.arange(10) % 2

idx = rng.permutation(len(X))
n_test = 3
test_idx, train_idx = idx[:n_test], idx[n_test:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print(X_test, y_test)

[[12 13]
 [18 19]
 [ 2  3]] [0 1 1]


Shuffling `X` and `y` separately would scramble which label belongs to which row, a bug that produces a model with exactly chance-level accuracy and no error message. Shuffling one index array and applying it to both is the safe pattern.

## 10. Putting it together: k-NN without loops

In the Module 1 mini project, k-NN made predictions with nested Python loops: for each test point, compute the distance to every training point, sort, and vote. Let's write it in NumPy and compare, on the same kind of data.

First, some data: three classes of points in 4 dimensions.

In [37]:
rng = np.random.default_rng(1)
centers = np.array([[0, 0, 0, 0], [3, 3, 0, 0], [0, 3, 3, 3]], dtype=float)

def make_data(n_per_class):
    X = np.vstack([rng.normal(c, 1.3, size=(n_per_class, 4)) for c in centers])
    y = np.repeat(np.arange(3), n_per_class)
    return X, y

X_train, y_train = make_data(700)
X_test, y_test = make_data(200)
print(X_train.shape, X_test.shape)

(2100, 4) (600, 4)


The plain Python version, as in Module 1:

In [38]:
from collections import Counter


def knn_python(X_train, y_train, X_test, k=5):
    train = [(list(map(float, x)), int(label)) for x, label in zip(X_train, y_train)]
    preds = []
    for x in X_test:
        x = list(map(float, x))
        dists = sorted(
            (sum((a - b) ** 2 for a, b in zip(x, xt)), label) for xt, label in train
        )
        votes = Counter(label for _, label in dists[:k])
        preds.append(min(votes, key=lambda c: (-votes[c], c)))
    return np.array(preds)

And the NumPy version. To keep memory use down, we compute all the distances with a well-known expansion instead of building a `(n_test, n_train, d)` array of differences:

$$\|a - b\|^2 = \|a\|^2 - 2\, a \cdot b + \|b\|^2$$

The middle term for every pair at once is a single matrix multiplication, `X_test @ X_train.T`, and the result is only `(n_test, n_train)`.

In [39]:
def knn_numpy(X_train, y_train, X_test, k=5):
    sq_dists = (
        (X_test ** 2).sum(axis=1)[:, None]          # (n_test, 1)
        - 2 * X_test @ X_train.T                     # (n_test, n_train)
        + (X_train ** 2).sum(axis=1)[None, :]        # (1, n_train)
    )
    nearest = np.argpartition(sq_dists, k, axis=1)[:, :k]    # indexes of the k smallest per row
    neighbour_labels = y_train[nearest]                      # (n_test, k)
    n_classes = y_train.max() + 1
    votes = (neighbour_labels[:, :, None] == np.arange(n_classes)).sum(axis=1)   # (n_test, n_classes)
    return votes.argmax(axis=1)

Three new ideas in there:

- **`np.argpartition(a, k)`** puts the `k` smallest values first (in no particular order) without fully sorting. That's all k-NN needs, and it's faster than `argsort` for large arrays.
- **`y_train[nearest]`** uses fancy indexing with a 2D index array, so it returns the labels of every neighbour of every test point in one step.
- **The vote** compares each neighbour label against every class number with broadcasting, `(n_test, k, 1) == (n_classes,)`, and counts matches along the neighbour axis. `argmax` picks the class with the most votes, and on a tie it picks the smallest class number, the same tie rule as the Python version.

Now the comparison:

In [40]:
start = time.perf_counter()
pred_py = knn_python(X_train, y_train, X_test)
t_py = time.perf_counter() - start

start = time.perf_counter()
pred_np = knn_numpy(X_train, y_train, X_test)
t_np = time.perf_counter() - start

print(f"python: {t_py:.3f} s   accuracy {(pred_py == y_test).mean():.1%}")
print(f"numpy:  {t_np:.4f} s   accuracy {(pred_np == y_test).mean():.1%}")
print(f"same predictions: {(pred_py == pred_np).all()}")
print(f"numpy is about {t_py / t_np:.0f}x faster")

python: 1.180 s   accuracy 93.5%
numpy:  0.0220 s   accuracy 93.5%
same predictions: True
numpy is about 54x faster


Identical predictions, and the NumPy version is dozens of times faster on this data (the exact number above depends on your machine, and the gap grows with the size of the dataset). It's also shorter, and once you're used to reading shapes, arguably clearer: every line is one whole step of the algorithm, applied to every point at once.

This is the pattern for the rest of the course, and for ML in general: understand an algorithm with loops, then express it with arrays.

## Summary

- NumPy arrays store numbers of one type contiguously, and run operations in compiled code over the whole array. Don't loop over them element by element in Python.
- Shape is everything. A dataset is `(n_samples, n_features)`. Use `reshape(-1, ...)`, `[:, None]` and `.T` to get the shapes you need.
- Basic slices are **views** that share memory with the original. Fancy and boolean indexing make copies. Use `.copy()` when you need independence.
- Boolean masks replace `if` statements: `x[x < 0] = 0`, `(pred == y).mean()`, `np.where(cond, a, b)`. Use `&`, `|`, `~` with parentheses.
- In reductions, `axis` is the dimension that disappears. `axis=0` gives per-feature statistics. `keepdims=True` keeps shapes compatible.
- Broadcasting matches shapes from the right; size-1 dimensions stretch. `(X - X.mean(axis=0)) / X.std(axis=0)` standardizes everything at once.
- Watch out for integer overflow (convert images to float first), float32 vs float64, NaN spreading (`np.nanmean`), and use `np.isclose`/`np.allclose` to compare.
- `@` is matrix multiplication: `(n, k) @ (k, m) -> (n, m)`. `*` is element-wise. Use `np.linalg.solve` or `lstsq`, never `inv`, to solve linear systems.
- Use `rng = np.random.default_rng(seed)`, and shuffle with one shared permutation of indexes.

Next, open `02_exercises.ipynb`.